# 02. 독립변수 — GTX 경쟁 목적지 비율

**원본**: `Analisis3/04_gtx_competing_destination_share_250m_morning.py`

- 정의: 2024년 07~09시 서울행 승차 중 **연신내역·서울역 1km 이내에서 하차**한 승차 비율(%)
- 방법: 승차 좌표를 250m 격자 폴리곤에 공간결합
- 2024년 오전 승차가 0명인 셀(14개)은 비율을 정의할 수 없어서 결측이다 (04에서 0으로 채우고 플래그를 남김)
- 원본과 132/132 셀 일치

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # 정리본 최상위의 config.py

from config import *
setup_notebook()

In [2]:
import geopandas as gpd

city_grid = stable_grid_polygons()
raw = read_csv(RAW_PATHS[2024])
ride_hour = pd.to_datetime(raw["ride_dt"].astype(str), format="%Y%m%d%H%M%S", errors="coerce").dt.hour
trips = pd.DataFrame({
    "board_lat": pd.to_numeric(raw["승차위도"], errors="coerce"),
    "board_lon": pd.to_numeric(raw["승차경도"], errors="coerce"),
    "alight_lat": pd.to_numeric(raw["하차위도"], errors="coerce"),
    "alight_lon": pd.to_numeric(raw["하차경도"], errors="coerce"),
    "boarding": pd.to_numeric(raw["utztn_nope"], errors="coerce").fillna(0),
    "board_hour": ride_hour,
    "alight_city": pd.to_numeric(raw["goff_ctpv_cd"], errors="coerce"),
})
trips = trips[trips["board_hour"].between(*MORNING_HOURS) & (trips["alight_city"] == SEOUL)].dropna(subset=["board_lat", "board_lon"])

points = gpd.GeoDataFrame(trips, geometry=gpd.points_from_xy(trips["board_lon"], trips["board_lat"]), crs="EPSG:4326").to_crs(city_grid.crs)
joined = gpd.sjoin(points, city_grid, how="inner", predicate="within").drop(columns=["index_right"])
joined["competitive"] = is_competing_destination(joined["alight_lat"], joined["alight_lon"])
print(f"07~09시 서울행 기록: {len(trips):,}건 → 안정 셀에 매칭: {len(joined):,}건")

07~09시 서울행 기록: 3,700건 → 안정 셀에 매칭: 2,407건


In [3]:
total = joined.groupby("셀 ID")["boarding"].sum()
competitive = joined.loc[joined["competitive"]].groupby("셀 ID")["boarding"].sum()

result = pd.DataFrame({"셀 ID": sorted(city_grid["셀 ID"])})
result["2024년 07~09시 전체 서울행 승차인원"] = result["셀 ID"].map(total).fillna(0)
result["2024년 07~09시 GTX 경쟁 목적지 승차인원"] = result["셀 ID"].map(competitive).fillna(0)
total_users = result["2024년 07~09시 전체 서울행 승차인원"]
result["GTX 경쟁 목적지 승차인원 비율(%)"] = np.where(
    total_users > 0, result["2024년 07~09시 GTX 경쟁 목적지 승차인원"] / total_users * 100, np.nan)
result.to_csv(COMPETING_MORNING_PATH, index=False, encoding="utf-8-sig")
print(f"비율 결측(2024 오전 승차 0명): {result.iloc[:, -1].isna().sum()}개 | 저장: {COMPETING_MORNING_PATH.name}")
result.iloc[:, -1].describe().round(1)

비율 결측(2024 오전 승차 0명): 14개 | 저장: 05_GTX_경쟁목적지비율_오전.csv


count    118.0
mean      15.6
std       26.3
min        0.0
25%        0.0
50%        0.0
75%       19.0
max      100.0
Name: GTX 경쟁 목적지 승차인원 비율(%), dtype: float64